In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/customer_support_tickets.csv")

df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [4]:
df_clean = df.copy()

df_clean.shape

(28587, 16)

In [5]:
df_clean["version"].value_counts()

version
400    18599
52      9119
51       869
Name: count, dtype: int64

In [6]:
df_clean.groupby("version")["language"].value_counts()

version  language
51       en            551
         de            318
52       en           5346
         de           3773
400      en          10441
         de           8158
Name: count, dtype: int64

In [12]:
print(df_clean.columns.tolist())
df_clean.shape



['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


(28587, 15)

In [13]:
df_clean = df_clean[df_clean["language"] == "en"].copy()

In [14]:
df_clean.shape

(16338, 15)

In [15]:
df_clean["language"].value_counts()

language
en    16338
Name: count, dtype: int64

In [16]:
df_clean.isnull().sum()

subject      2607
body            0
answer          3
type            0
queue           0
priority        0
language        0
tag_1           0
tag_2           6
tag_3          69
tag_4        1711
tag_5        7922
tag_6       12968
tag_7       15204
tag_8       16057
dtype: int64

In [17]:
df_clean = df_clean.dropna(subset=["answer"])

In [18]:
df_clean["subject"] = df_clean["subject"].fillna("")

In [19]:
df_clean.isnull().sum()

subject         0
body            0
answer          0
type            0
queue           0
priority        0
language        0
tag_1           0
tag_2           6
tag_3          69
tag_4        1709
tag_5        7920
tag_6       12965
tag_7       15201
tag_8       16054
dtype: int64

In [20]:
df_clean = df_clean.drop(
    columns=[
        "tag_1","tag_2","tag_3","tag_4",
        "tag_5","tag_6","tag_7","tag_8"
    ]
)

In [21]:
df_clean.columns

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language'], dtype='object')

In [22]:
df_clean = df_clean.drop(columns=["language"])

In [23]:
df_clean.columns

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority'], dtype='object')

In [24]:
df_clean["ticket_text"] = (
    df_clean["subject"] + " " + df_clean["body"]
)

In [25]:
df_clean[["ticket_text"]].head()

,ticket_text
1,"Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...
5,"Feature Query Dear Customer Support,\n\nI hope..."


In [33]:
import re
import string

def clean_text(text):
    text = text.lower()

    # Replace literal \n with space
    text = text.replace("\\n", " ")

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_clean["clean_text"] = df_clean["ticket_text"].apply(clean_text)

df_clean[["ticket_text", "clean_text"]].head()

,ticket_text,clean_text
1,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support team ...
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
5,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer support i hope thi...


In [34]:
df_clean.to_csv("../data/customer_support_tickets_clean.csv", index=False)